<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
.definition {border-left:5px solid #7c3aed;background:#f5f3ff;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## TW15 · Session 1 · K-Means and Hierarchical Clustering

        **Course level:** developing beginner  
        **Prior learning:** feature scaling, pandas summaries, and supervised-learning evaluation  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        Unsupervised learning searches for structure without a target label. We cluster synthetic customers, evaluate separation, and interpret groups in the original business units.

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - explain the aim and limitations of clustering
- scale features before distance-based analysis
- fit and interpret a k-means model
- use inertia and silhouette score to compare k values
- build hierarchical clusters and read a dendrogram


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Retrieval | 10 min | Recall earlier Python and data skills |
| New concepts | 30 min | Learn with short, explained examples |
| Guided analysis | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation, predict the result, run the cell with <code>Shift + Enter</code>, then change one thing. Edit only cells marked <strong>Student task</strong> unless your lecturer says otherwise.</div>


### Load the customer-segmentation data

The reference segment is held aside and is not given to either clustering algorithm.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

_candidates = [
    Path.cwd() / "data",
    Path.cwd() / "Python_Data_Science_TW09_TW17" / "data",
]
DATA_DIR = next((path for path in _candidates if path.exists()), _candidates[0])
if not DATA_DIR.exists():
    raise FileNotFoundError("Data folder not found. Start JupyterLab from the extracted course folder.")
print("Data folder:", DATA_DIR.resolve())

customers = pd.read_csv(DATA_DIR / "customer_segments.csv")
feature_columns = [
    "annual_income", "annual_spend",
    "visits_per_month", "digital_engagement",
]
X = customers[feature_columns]
reference = customers["reference_segment"]
print(customers.head())
print(X.describe().round(1))


## 1. What clustering can and cannot do

Clustering groups observations according to a chosen representation and distance. A cluster is not automatically a natural, permanent, or actionable customer type. Results depend on feature choice, scaling, algorithm, and parameters.


### Check feature scales

Income values are much larger numerically than visit counts. Without scaling, income would dominate Euclidean distance.


In [ ]:
scale_comparison = pd.DataFrame({
    "minimum": X.min(),
    "maximum": X.max(),
    "standard_deviation": X.std(),
})
print(scale_comparison.round(1))


### Standardise the features

Each transformed feature has approximately zero mean and unit standard deviation.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
scaled_frame = pd.DataFrame(X_scaled, columns=feature_columns)

print(scaled_frame.mean().round(3))
print(scaled_frame.std(ddof=0).round(3))


## 2. K-means

K-means alternates between assigning points to their nearest centroid and updating centroids. It seeks compact groups under squared Euclidean distance and requires the number of clusters k in advance.


### Fit a three-cluster solution

random_state makes initialisation reproducible; multiple starts reduce sensitivity to one unlucky start.


In [ ]:
kmeans = KMeans(n_clusters=3, n_init=20, random_state=42)
kmeans_labels = kmeans.fit_predict(X_scaled)

print("Cluster counts:")
print(pd.Series(kmeans_labels).value_counts().sort_index())
print("Inertia:", round(kmeans.inertia_, 2))
print("Silhouette:", round(silhouette_score(X_scaled, kmeans_labels), 3))


## 3. Compare possible k values

Inertia always falls as k increases, so there is no automatic best minimum. Look for an elbow and combine it with silhouette score, stability, interpretability, and intended use.


### Calculate inertia and silhouette for k=2 through 7

Silhouette values closer to 1 indicate compact, separated groups under the chosen distance.


In [ ]:
k_results = []
for k in range(2, 8):
    candidate = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = candidate.fit_predict(X_scaled)
    k_results.append({
        "k": k,
        "inertia": candidate.inertia_,
        "silhouette": silhouette_score(X_scaled, labels),
    })

k_table = pd.DataFrame(k_results)
print(k_table.round(3))


### Plot the elbow and silhouette evidence

Two panels prevent metrics with different units from sharing a confusing axis.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(k_table["k"], k_table["inertia"], marker="o")
axes[0].set_title("Elbow evidence")
axes[0].set_xlabel("Number of clusters")
axes[0].set_ylabel("Inertia")
axes[1].plot(k_table["k"], k_table["silhouette"], marker="o", color="#d97706")
axes[1].set_title("Silhouette evidence")
axes[1].set_xlabel("Number of clusters")
axes[1].set_ylabel("Silhouette score")
fig.tight_layout()
plt.show()
plt.close(fig)


## 4. Interpret in original units

Cluster numbers are arbitrary labels. Profile each cluster using original features, sample size, and variability before assigning a descriptive business name.


### Create cluster profiles

Medians resist unusual observations; means and counts provide additional context.


In [ ]:
profiled = customers.copy()
profiled["cluster"] = kmeans_labels
cluster_profile = profiled.groupby("cluster").agg(
    customers=("customer_id", "count"),
    median_income=("annual_income", "median"),
    median_spend=("annual_spend", "median"),
    median_visits=("visits_per_month", "median"),
    median_digital=("digital_engagement", "median"),
)
print(cluster_profile.round(1))


### Visualise two original dimensions

This projection cannot show every feature, so it supports rather than proves cluster quality.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    customers["annual_income"],
    customers["annual_spend"],
    c=kmeans_labels,
    cmap="viridis",
    alpha=0.7,
)
ax.set_title("K-means customer groups")
ax.set_xlabel("Annual income (£)")
ax.set_ylabel("Annual spend (£)")
ax.legend(*scatter.legend_elements(), title="Cluster")
fig.tight_layout()
plt.show()
plt.close(fig)


## 5. Hierarchical clustering

Agglomerative clustering begins with individual observations and repeatedly merges the closest groups. A dendrogram visualises the merge sequence and distances for a selected linkage method.


### Draw a dendrogram

Ward linkage seeks merges that limit within-cluster variance. truncate_mode keeps the teaching figure readable.


In [ ]:
linkage_matrix = linkage(X_scaled, method="ward")
fig, ax = plt.subplots(figsize=(10, 5))
dendrogram(
    linkage_matrix,
    truncate_mode="lastp",
    p=20,
    leaf_rotation=45,
    leaf_font_size=9,
    ax=ax,
)
ax.set_title("Truncated hierarchical-clustering dendrogram")
ax.set_xlabel("Merged groups")
ax.set_ylabel("Ward distance")
fig.tight_layout()
plt.show()
plt.close(fig)


### Create three agglomerative clusters

Adjusted Rand Index compares two label assignments while ignoring arbitrary label numbers.


In [ ]:
hierarchical = AgglomerativeClustering(n_clusters=3, linkage="ward")
hierarchical_labels = hierarchical.fit_predict(X_scaled)

agreement = adjusted_rand_score(kmeans_labels, hierarchical_labels)
print("Hierarchical counts:")
print(pd.Series(hierarchical_labels).value_counts().sort_index())
print("Agreement with k-means (ARI):", round(agreement, 3))


## Guided recommendation

An actionable segment should be sufficiently large, stable, interpretable, reachable by available operations, and assessed for fairness. A marketing label must not imply sensitive traits or certainty unsupported by the data.


### Create cautious segment descriptions

Descriptions are based on medians and remain provisional.


In [ ]:
cluster_order = cluster_profile["median_spend"].sort_values().index.tolist()
segment_names = {
    cluster_order[0]: "Lower observed engagement",
    cluster_order[1]: "Moderate observed engagement",
    cluster_order[2]: "Higher observed engagement",
}
profiled["segment_name"] = profiled["cluster"].map(segment_names)

print(profiled["segment_name"].value_counts())
print("Next step: test stability and business usefulness before deployment.")


## Student coding lab

Use only feature columns for fitting; keep identifiers and reference labels separate.


        ### Student task 1: Prepare clustering features

        <div class="task"><strong>Your job:</strong> Select the four numerical features, check missingness and scale, standardise them, and verify transformed means and standard deviations.</div>

        **Check your work**

        - Identifiers and reference labels are excluded.
- Missingness is checked.
- Standardised means and deviations are verified.


In [ ]:
# STUDENT TASK 1
task_X = pd.DataFrame()
task_X_scaled = np.empty((0, 0))
print(task_X.shape, task_X_scaled.shape)


        ### Student task 2: Compare k values

        <div class="task"><strong>Your job:</strong> For k=2 through 8, calculate inertia and silhouette with 20 starts and random_state 123.</div>

        **Check your work**

        - Each k uses the same data and random state.
- Both metrics are stored.
- A proposed k uses evidence rather than inertia alone.


In [ ]:
# STUDENT TASK 2
task_k_results = []
print(pd.DataFrame(task_k_results))


        ### Student task 3: Profile chosen clusters

        <div class="task"><strong>Your job:</strong> Fit a chosen k-means solution and report count, mean, median, and standard deviation for every original feature by cluster.</div>

        **Check your work**

        - Labels come from fitted k-means.
- Profiles use original units.
- Labels are not given ordinal meaning.


In [ ]:
# STUDENT TASK 3
chosen_k = 3
task_labels = np.zeros(len(customers), dtype=int)
task_profile = pd.DataFrame()
print(task_profile)


        ### Student task 4: Create hierarchical clusters

        <div class="task"><strong>Your job:</strong> Build Ward-linkage agglomerative clusters using the same k and compare them with k-means using adjusted Rand index.</div>

        **Check your work**

        - The same scaled matrix is used.
- Cluster counts are displayed.
- ARI interpretation recognises that label numbers are arbitrary.


In [ ]:
# STUDENT TASK 4
hierarchical_task_labels = np.zeros(len(customers), dtype=int)
algorithm_agreement = 0.0
print("ARI:", algorithm_agreement)


        ### Student task 5: Write a segment recommendation

        <div class="task"><strong>Your job:</strong> Create descriptive names from profiles and document size, defining evidence, possible action, limitation, and fairness risk for each.</div>

        **Check your work**

        - Descriptions are evidence-based and non-judgemental.
- Actions do not assume causality.
- Every segment has a stated risk or limitation.



**Optional extension:** Repeat clustering after removing one feature and quantify assignment stability.


In [ ]:
# STUDENT TASK 5
segment_recommendations = pd.DataFrame({
    "cluster": [],
    "description": [],
    "possible_action": [],
    "limitation": [],
    "fairness_risk": [],
})
print(segment_recommendations)


        ## Knowledge check

        Answer these without running new code first.

        1. Why should these features be scaled?
2. Why does inertia always fall as k increases?
3. What does silhouette score assess?
4. Why are cluster numbers arbitrary?
5. How does hierarchical clustering differ from k-means?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** Session 2 introduces PCA and t-SNE and validates unsupervised results through stability and reference comparisons.
